# VAE variants (beta-VAE, VQ-VAE) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## latent pressure in VAE variants
The VAE objective from 10.2 supplies the reconstruction-versus-KL trade. Information bottlenecks and clustering explain why changing that pressure can produce disentangled or discrete representations. These ideas feed tokenized image generation, latent diffusion (10.14), and conditional control (10.15).

In [ ]:
# 1) beta turns the KL knob in beta-VAE
betas=np.linspace(0,6,80); recon=0.42; kl=0.2482926519; loss=recon+betas*kl
plt.figure(figsize=(5,3)); plt.plot(betas,loss); plt.xlabel('beta'); plt.ylabel('loss'); plt.title('larger beta prices information harder')
assert abs(loss[0]-0.42)<1e-9 and loss[-1]>loss[0]

In [ ]:
# 2) the best latent code changes as beta changes
recon=np.array([0.30,0.42,0.70]); kl=np.array([0.80,0.2482926519,0.02]); beta_vals=[0.2,1,4]; best=[int(np.argmin(recon+b*kl)) for b in beta_vals]
plt.figure(figsize=(5,3)); plt.plot(beta_vals,best,'o-'); plt.yticks([0,1,2]); plt.xlabel('beta'); plt.title('higher beta picks lower-KL code')
assert best[0]<best[-1]

In [ ]:
# 3) VQ-VAE nearest codebook lookup
z=np.array([0.2,0.7]); code=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]]); d=np.linalg.norm(code-z,axis=1)
plt.figure(figsize=(4,4)); plt.scatter(code[:,0],code[:,1],s=80); plt.scatter([z[0]],[z[1]],c='r'); plt.title('nearest codebook vector')
assert int(np.argmin(d))==1

In [ ]:
# 4) too few codebook entries quantize coarsely
pts=np.random.default_rng(0).normal(size=(80,2))*0.2+np.array([0.2,0.7]); q=code[np.argmin(((pts[:,None,:]-code[None,:,:])**2).sum(-1),axis=1)]
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1],s=10); plt.scatter(q[:,0],q[:,1],s=10); plt.title('continuous points snap to codes')
assert len(np.unique(q,axis=0))>=1

In [ ]:
# 5) commitment loss pulls encoder outputs toward chosen code
chosen=np.array([0.,1.]); alphas=np.linspace(0,1,20); path=(1-alphas[:,None])*z+alphas[:,None]*chosen; loss=np.sum((path-chosen)**2,axis=1)
plt.figure(figsize=(4,4)); plt.plot(path[:,0],path[:,1]); plt.scatter([chosen[0]],[chosen[1]],c='r'); plt.title('commitment pulls toward code')
assert loss[-1]<loss[0]